# π₀ + PaliGemma — experimental baseline

This is the **real** π₀: PaliGemma 3B as the VLM backbone, our flow-matching action expert on top. Establishes the experimental baseline against which every later paper in the arc is compared.

Paper: `papers/2024-10-31_pi0_first-generalist-policy.pdf`

## What this notebook does

1. Loads the PaliGemma processor (cheap; verifies HF auth).
2. Attempts to load PaliGemma 3B (~6 GB bf16). Falls back to a clear message if memory runs out.
3. Builds a `Pi0Policy` with the real backbone via `Pi0Policy.from_pretrained(PALIGEMMA_3B)`.
4. Runs one forward pass; samples one flow-matching action chunk.
5. Runs one KI training step (discrete CE on the VLM + continuous flow-matching MSE on the expert) with the stop-gradient applied at the interface.

## Memory expectations

| Device | Need | Notes |
|---|---|---|
| Apple Silicon MPS, 16 GB unified | tight; should work bf16 | first forward might be slow due to MPS op fallback |
| Apple Silicon MPS, 32+ GB unified | comfortable | recommended local |
| 1× A100 80 GB on RunPod | comfortable | the default dev pod |
| 1× H100 80 GB on RunPod | very comfortable | fastest |

If your laptop can't hold PaliGemma, that's fine — push to git and run this on the dev pod (`docs/runpod.md`).

In [ ]:
import time
import torch
from PIL import Image
import numpy as np

from pi_stack.models.backbones import PALIGEMMA_3B, load_backbone
from pi_stack.models.pi0 import Pi0Config, Pi0Policy

# Pick the strongest device we have.
if torch.cuda.is_available():
    DEVICE = 'cuda'
    DTYPE = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    DTYPE = torch.bfloat16   # PaliGemma ships bf16; MPS supports it
else:
    DEVICE = 'cpu'
    DTYPE = torch.float32
print(f'device : {DEVICE}')
print(f'dtype  : {DTYPE}')

## 1. Processor only — fastest sanity check

Loads ~5 MB of processor config. If this fails, your HF auth is broken (run `huggingface-cli whoami` to debug). If it succeeds, you're set up; the full model is the next gate.

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(PALIGEMMA_3B.hf_repo)
print('processor type :', type(processor).__name__)
print('image size     :', processor.image_processor.size)
print('vocab size     :', processor.tokenizer.vocab_size)

## 2. Full model load — the expensive one

First call downloads ~6 GB (cached at `~/.cache/huggingface`). Subsequent runs load from cache in seconds.

On a fresh laptop this can OOM. The cell wraps everything in try/except so you get a clean error rather than a crashed kernel.

In [ ]:
policy = None
try:
    t0 = time.perf_counter()
    policy = Pi0Policy.from_pretrained(
        PALIGEMMA_3B,
        config=Pi0Config(backbone=PALIGEMMA_3B, state_dim=14, image_resolution=224),
        device=DEVICE,
        dtype=DTYPE,
    )
    t_load = time.perf_counter() - t0
    print(f'loaded in {t_load:.1f} s')
    if DEVICE == 'cuda':
        peak_gb = torch.cuda.max_memory_allocated() / (1024**3)
        print(f'CUDA peak after load: {peak_gb:.2f} GB')
    n_params = sum(p.numel() for p in policy.parameters())
    print(f'total params         : {n_params:,d}')
    n_backbone = sum(p.numel() for p in policy.backbone.parameters())
    n_expert = sum(p.numel() for p in policy.action_expert.parameters())
    n_state = sum(p.numel() for p in policy.state_encoder.parameters())
    print(f'  backbone (PaliGemma): {n_backbone:>13,d}')
    print(f'  state encoder       : {n_state:>13,d}')
    print(f'  action expert       : {n_expert:>13,d}')
except Exception as e:
    print('LOAD FAILED:', type(e).__name__)
    print(str(e)[:500])
    print()
    print('Most likely cause: insufficient memory for PaliGemma 3B bf16 (~6 GB).')
    print('Fix: run this on a RunPod dev pod (docs/runpod.md).')
    print('Workaround for this notebook: skip to §5, which uses TinyBackbone instead.')

## 3. One forward pass

Build a dummy gray image + a short prompt. Use the processor to tokenize properly (PaliGemma's input requires its `<image>...<bos>prompt\n` layout).

In [ ]:
if policy is None:
    print('skipped: policy did not load')
else:
    image = Image.new('RGB', (224, 224), color=(128, 128, 128))
    inputs = policy.backbone.preprocess(text='pick up the red cube', images=image)
    input_ids   = inputs['input_ids'].to(DEVICE)
    pixel_values = inputs['pixel_values'].to(DEVICE).to(DTYPE)
    state = torch.zeros(1, 14, device=DEVICE, dtype=DTYPE)

    # Time the action chunk sample.
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        chunk = policy.predict_chunk(
            images=pixel_values, state=state, language_ids=input_ids,
        )
    if DEVICE == 'cuda': torch.cuda.synchronize()
    print(f'chunk shape   : {tuple(chunk.shape)} dtype={chunk.dtype}')
    print(f'forward+sample: {(time.perf_counter() - t0)*1000:.1f} ms')
    print(f'action range  : [{chunk.float().min().item():+.3f}, {chunk.float().max().item():+.3f}]')

## 4. KI training step

One pass of the **Knowledge Insulation** recipe (`pi_stack.training.ki`) with the *real* backbone in the loop:

- VLM forward → discrete cross-entropy on FAST-ish target tokens.
- Action expert forward (with stop-gradient on VLM features) → flow-matching MSE.
- Combined loss, single `.backward()` step.

We assert the VLM's gradients are non-zero (its discrete loss path) and the action expert's gradients are non-zero. If the stop-gradient were broken, the action expert loss would also contribute to the VLM gradient — but KI's `insulate()` prevents that.

In [ ]:
if policy is None:
    print('skipped: policy did not load')
else:
    from pi_stack.training.ki import KIConfig, ki_loss, insulate, flow_matching_target

    # Same inputs as before.
    image = Image.new('RGB', (224, 224), color=(128, 128, 128))
    inputs = policy.backbone.preprocess(text='pick up the red cube', images=image)
    input_ids = inputs['input_ids'].to(DEVICE)
    pixel_values = inputs['pixel_values'].to(DEVICE).to(DTYPE)
    state = torch.zeros(1, 14, device=DEVICE, dtype=DTYPE)

    # 1) Backbone forward — get logits and features.
    logits, features = policy.backbone(input_ids, images=pixel_values)
    print('logits shape    :', tuple(logits.shape))
    print('features shape  :', tuple(features.shape))

    # 2) KI stop-gradient at the interface.
    ctx = insulate(features, enabled=True)
    # Append the state token so the action expert sees the full π₀ context.
    ctx = torch.cat([ctx, policy.state_encoder(state)], dim=1)

    # 3) Flow-matching forward through the action expert.
    horizon = policy.action_expert.config.horizon
    action_dim = policy.action_expert.config.action_dim
    actions = torch.randn(1, horizon, action_dim, device=DEVICE, dtype=DTYPE)
    noise = torch.randn_like(actions)
    t = torch.rand(1, device=DEVICE, dtype=DTYPE)
    a_t, target = flow_matching_target(actions, noise, t)
    v_pred = policy.action_expert(a_t, t, ctx)

    # 4) Combined KI loss.
    # Discrete target: random token ids (placeholder for FAST-tokenized actions).
    discrete_targets = torch.randint(0, PALIGEMMA_3B.vocab_size, input_ids.shape, device=DEVICE)
    out = ki_loss(
        discrete_logits=logits.float(),         # CE wants float
        discrete_targets=discrete_targets,
        continuous_pred=v_pred.float(),
        continuous_target=target.float(),
        config=KIConfig(),
    )
    print()
    print(f'discrete loss   : {out["discrete"].item():.3f}')
    print(f'continuous loss : {out["continuous"].item():.4f}')
    print(f'combined loss   : {out["loss"].item():.3f}')

    out['loss'].backward()
    backbone_grad = sum((p.grad.float()**2).sum().item() for p in policy.backbone.parameters() if p.grad is not None) ** 0.5
    expert_grad   = sum((p.grad.float()**2).sum().item() for p in policy.action_expert.parameters() if p.grad is not None) ** 0.5
    print()
    print(f'||grad backbone||  : {backbone_grad:.4e}   (from discrete loss; KI keeps it isolated)')
    print(f'||grad expert||    : {expert_grad:.4e}   (from continuous loss)')

## 5. Local fallback — TinyBackbone

If the PaliGemma load failed, the rest of the π₀ pipeline still runs locally on the in-repo `TinyBackbone`. Same code path; only the `backbone=` differs.

In [ ]:
from pi_stack.models.backbones import TINY
tiny_policy = Pi0Policy(Pi0Config(backbone=TINY, state_dim=14, image_resolution=64))
B = 2
tiny_chunk = tiny_policy.predict_chunk(
    images=torch.randn(B, 3, 64, 64),
    state=torch.randn(B, 14),
    language_ids=torch.randint(0, TINY.vocab_size, (B, 6)),
)
print('TinyBackbone chunk:', tuple(tiny_chunk.shape))

## Takeaways

- The π₀ assembly is `Pi0Policy.from_pretrained(PALIGEMMA_3B, device=...)`. One line.
- Forward pass returns a flow-matching-sampled action chunk shaped `(B, H, D)`.
- The KI recipe in `pi_stack.training.ki` plugs in unchanged — `insulate()` blocks the VLM gradient from the action-expert loss.
- Swap to `TinyBackbone` for the inner-loop dev cycle; back to `PALIGEMMA_3B` for the real experimental baseline.

Next steps if this notebook runs cleanly:
- Wire `scripts/train.py` to dispatch a real KI training loop on Libero / OXE (§4 of `CHECKLIST.md`).
- Drop the real backbone into `Pi05Policy` / `Pi06Policy` / `Pi07Policy` — the inheritance chain already accepts any backbone that satisfies the `(logits, features) + encode_image_features` contract.